
**Phase 1 Data Quality & Sanity Checks**

Before ANY statistical analysis, we answer 4 questions:
1. Did randomization work?           -> SRM (chi-square on group sizes)
2. Are users on the right page?      -> group vs landing_page cross-tab
3. Are users in only one group?      -> duplicate user_id check
4. Are baseline attributes balanced? -> covariate balance check

If ANY check fails, the experiment cannot be analyzed for ship/no-ship without
remediation. 


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
print(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))


c:\Users\ruthr\OneDrive\Documents\Claude\Projects\Github DS Portfolio Projects\01-ab-test-analysis


In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
# from src.data.loader import load_ab_data
from src.analysis.sanity_checks import check_srm


---------------------------------------------------------------------
1. Load the data
---------------------------------------------------------------------

In [3]:
df = pd.read_csv("../data/experiment.csv", parse_dates=["timestamp"])
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}\n")
display(df.head())

Shape: (100000, 14)
Columns: ['user_id', 'timestamp', 'group', 'landing_page', 'country', 'device', 'source', 'is_returning', 'prior_watch_hours', 'converted', 'trial_watch_hours', 'distinct_titles', 'day7_active', 'page_load_ms']



,user_id,timestamp,group,landing_page,country,device,source,is_returning,prior_watch_hours,converted,trial_watch_hours,distinct_titles,day7_active,page_load_ms
0,1029790,2026-04-01 00:00:20,treatment,personalized,US,Web,organic,False,0.02,0,8.79,5,0,503
1,1082885,2026-04-01 00:00:31,control,top_picks,UK,Web,organic,False,0.01,0,4.75,2,0,495
2,1037217,2026-04-01 00:01:44,treatment,personalized,Other,Android,paid_social,False,0.00,0,0.48,0,0,363
3,1048257,2026-04-01 00:01:45,treatment,personalized,CA,Web,organic,False,0.00,1,6.99,5,1,497
4,1053311,2026-04-01 00:01:50,control,top_picks,US,Web,paid_search,False,0.07,0,1.20,0,0,426


---------------------------------------------------------------------
2. SRM check - Did randomization work?
---------------------------------------------------------------------
We expect a 50/50 split. We use the very strict alpha = 0.001 because
false positives are costly (we'd block a valid experiment).

In [18]:
srm = check_srm(df, group_col="group", expected_ratio={"control": 0.5, "treatment": 0.5})
print("\n=== SRM check ===")
print(srm)


print("\n====VERDICT====")
print(""" 
At α = 0.001 (industry-strict): SRM check PASS — but borderline.
At the conventional α = 0.01 or 0.05 thresholds this would be flagged. 
A p-value of 0.0036 means there is a 1-in-280 chance of seeing an imbalance 
this large under true 50/50 randomization — small enough to warrant a closer 
look before trusting downstream results. Treatment is under-represented by ~922 
users(~0.92pp). This is informative: it suggests the assignment system is 
losing treatment users somewhere — not gaining control users.""")


print("\n=== Next Steps ===")
print("""
Follow up with covariate-level balance checks (Phase 1b) to 
localize the imbalance to a specific segment. If a segment is identified, 
decide between (a) excluding the affected subset and analyzing the rest, 
or (b) running ITT and disclosing the limitation in the decision memo.""")


=== SRM check ===
✅ PASS
  Observed: {'control': 50461, 'treatment': 49539}
  Expected: {'control': 50000.0, 'treatment': 50000.0}
  Chi-square: 8.5008, p-value: 0.003550

====VERDICT====
 
At α = 0.001 (industry-strict): SRM check PASS — but borderline.
At the conventional α = 0.01 or 0.05 thresholds this would be flagged. 
A p-value of 0.0036 means there is a 1-in-280 chance of seeing an imbalance 
this large under true 50/50 randomization — small enough to warrant a closer 
look before trusting downstream results. Treatment is under-represented by ~922 
users(~0.92pp). This is informative: it suggests the assignment system is 
losing treatment users somewhere — not gaining control users.

=== Next Steps ===

Follow up with covariate-level balance checks (Phase 1b) to 
localize the imbalance to a specific segment. If a segment is identified, 
decide between (a) excluding the affected subset and analyzing the rest, 
or (b) running ITT and disclosing the limitation in the decision mem

---------------------------------------------------------------------
3. Assignment integrity - Are users on the right page?
---------------------------------------------------------------------
Treatment users should ONLY see 'personalized'. Control should ONLY see 'top_picks'.

In [5]:
print("\n=== Group vs landing_page cross-tab ===")
crosstab = df.groupby(["group", "landing_page"]).size().unstack(fill_value=0)
print(crosstab)


=== Group vs landing_page cross-tab ===
landing_page  personalized  top_picks
group                                
control                 48      50413
treatment            49474         65


In [6]:
mismatched = df[
    ((df["group"] == "control") & (df["landing_page"] != "top_picks")) |
    ((df["group"] == "treatment") & (df["landing_page"] != "personalized"))
]
mismatch_rate = len(mismatched) / len(df)
print(f"\nMismatched rows: {len(mismatched):,} ({mismatch_rate:.3%})")
print(" Interpretation: <0.5% is usually tolerable (caching, race conditions).")
print(" Higher than that = systemic issue. Decision: drop these rows OR")
print(" analyze as ITT (intent-to-treat, using assignment not actual exposure).")


Mismatched rows: 113 (0.113%)
 Interpretation: <0.5% is usually tolerable (caching, race conditions).
 Higher than that = systemic issue. Decision: drop these rows OR
 analyze as ITT (intent-to-treat, using assignment not actual exposure).


---------------------------------------------------------------------
4. Duplicate users - Are any users in BOTH groups?
---------------------------------------------------------------------

In [7]:
users_per_group = df.groupby("user_id")["group"].nunique().sort_values(ascending=False)
print(users_per_group.head())
multi_group = users_per_group[users_per_group > 1]
print(f"\n=== Duplicate users ===")
print(f"Users appearing in multiple groups: {len(multi_group):,}")
print("In real experiments this happens via: deleted cookies, multi-device,")
print("shared accounts. Common cleaning rules: drop them, or keep first.")

user_id
1000000    1
1066650    1
1066672    1
1066671    1
1066670    1
Name: group, dtype: int64

=== Duplicate users ===
Users appearing in multiple groups: 0
In real experiments this happens via: deleted cookies, multi-device,
shared accounts. Common cleaning rules: drop them, or keep first.


---------------------------------------------------------------------
5. Covariate balance check - Are baseline attributes balanced?
---------------------------------------------------------------------
Even if SRM passes, individual attribute distributions should be balanced
across arms (because randomization should be independent of attributes).

Covariate balance can capture subtle assignment failure which SRM alone could miss. 

For each user attribute (country, device, source, is_returning),
compute the % share in control vs treatment. Any large gap (>1pp absolute)
is a yellow flag.

In [ ]:
print("\n====== Categorical covariate balance check =======\n")
threshold_pp=0.01 # 1 percentage point absolute gap = yellow flag 
categorical=["country", "device", "source", "is_returning"]
for col in categorical:
    share = (df.groupby(["group", col]).size()
                / df.groupby("group").size()).unstack(level="group")
    share["gap_pp"] = (share["treatment"] - share["control"]).round(3)
    share["flag"] = share["gap_pp"].abs() > threshold_pp
    print(share.sort_values("gap_pp", key=abs, ascending=False))
    flag_maker=" Imbalance detected" if share["flag"].any() else " No imbalance detected"
    print(f"\n{col} distribution by group: {flag_maker}\n")
    print("=" * 70)
  
print("\n=== VERDICT ===")
print(
"   Look at the 'device' panel — the Android share is meaningfully\n"
"   lower in treatment vs. control. Combined with the borderline\n"
"   SRM result, this is consistent with an Android-specific\n"
"   assignment bug. Recommend: file engineering ticket, decide\n"
"   whether to (a) exclude affected users from analysis or\n"
"   (b) analyze ITT(Intent-to-Treat) and disclose the limitation in the memo."
)



====== Categorical covariate balance check =======

group     control  treatment  gap_pp   flag
country                                    
UK       0.151840   0.148630  -0.003  False
CA       0.100157   0.102364   0.002  False
US       0.447316   0.449081   0.002  False
Other    0.220150   0.219060  -0.001  False
AU       0.080537   0.080866   0.000  False

country distribution by group:  No imbalance detected

group     control  treatment  gap_pp   flag
device                                     
Android  0.355621   0.344718  -0.011   True
iOS      0.298884   0.304831   0.006  False
Web      0.244585   0.250005   0.005  False
TV       0.100910   0.100446  -0.000  False

device distribution by group:  Imbalance detected

group         control  treatment  gap_pp   flag
source                                         
organic      0.399239   0.398938    -0.0  False
paid_search  0.249301   0.249642     0.0  False
paid_social  0.202354   0.202245    -0.0  False
referral     0.149105   0.1

For the continuous covariate `prior_watch_hours`, compare means and run a
two-sample t-test (scipy.stats.ttest_ind). If p < 0.001 for a covariate,
there's likely an assignment leak.

In [9]:
print("\n====== Continuous covariate balance check =======\n")
print("=== Prior watch hours t-test ===")
from scipy import stats
c = df.loc[df.group == "control", "prior_watch_hours"]
t = df.loc[df.group == "treatment", "prior_watch_hours"]
t_stat, p_value = stats.ttest_ind(c, t,equal_var=False) # Welch's t-test (doesn't assume equal variances)
print(f"Control mean: {c.mean():.4f}, std: {c.std():.4f} n={len(c):,}")
print(f"Treatment mean: {t.mean():.4f}, std: {t.std():.4f} n={len(t):,}")
print(f"Mean difference (treatment - control): {(t.mean() - c.mean()):.4f}")
print(f"Welch t-test: t={t_stat:.4f}, p-value: {p_value:.4f}")
print(f"\n At alpha=0.001 (strict), this {'IS' if p_value < 0.001 else 'is NOT'} a significant imbalance.")


====== Continuous covariate balance check =======

=== Prior watch hours t-test ===
Control mean: 2.0907, std: 4.4024 n=50,461
Treatment mean: 2.1129, std: 4.4234 n=49,539
Mean difference (treatment - control): 0.0222
Welch t-test: t=-0.7964, p-value: 0.4258

 At alpha=0.001 (strict), this is NOT a significant imbalance.


**Key Points to Highlight**
* We know that randomization worked because SRM and covariate balance. SRM was broderline non-significant overall but the device covariate revealed an Android imbalance.
* Now we identified the imbalance to a specific segment, we would either exclude affected users from primary analysis and disclose the exclusion in the memo, or analyze under ITT and quantify how the leak biases the estimate. 
* The flag threshold we chose for this project is 1pp which is common industry rules of thumbs. But we can tighten it for high-stakes tests or relax for small samples. 